In [1]:
!pip install -q qiskit qiskit-ibm-provider qiskit-aer-gpu-cu11 qiskit-ibm-runtime pylatexenc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 82.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.9/249.9 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 39.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 17.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.3/111.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

In [ ]:
def reg(cells, idx):
    """
    Returns the two qubits corresponding to Sudoku cell `idx`.
    """
    return cells[2 * idx], cells[2 * idx + 1]

In [4]:
def initialize_search_space(qc, cells):
    """
    Apply Hadamard gates to every search qubit,
    creating an equal superposition over all states.
    """
    qc.h(cells)

In [5]:
def initialize_phase_ancilla(qc, phase):
    """
    Prepare phase ancilla in |->
    """
    qc.x(phase[0])
    qc.h(phase[0])

In [6]:
def encoding_check(qc, cell_reg, anc):
    """
    anc = 1 if encoding is valid
        = 0 if encoding == 11

    cell_reg = (MSB, LSB)
    """

    msb, lsb = cell_reg

    qc.ccx(msb, lsb, anc)
    qc.x(anc)

In [7]:
def undo_encoding_check(qc, cell_reg, anc):
    """
    Reverse encoding_check().
    """

    msb, lsb = cell_reg

    qc.x(anc)
    qc.ccx(msb, lsb, anc)

In [ ]:
def compare_registers(qc, regA, regB, cmp):
    """
    Compare two 2-qubit registers.

    cmp[0] : XNOR(MSB)
    cmp[1] : XNOR(LSB)
    cmp[2] : Equality flag

    Result:
        cmp[2] == 1 iff regA == regB
    """

    a1, a0 = regA
    b1, b0 = regB

    # Initialize XNOR ancillas
    qc.x(cmp[0])
    qc.x(cmp[1])

    # MSB XNOR
    qc.cx(a1, cmp[0])
    qc.cx(b1, cmp[0])

    # LSB XNOR
    qc.cx(a0, cmp[1])
    qc.cx(b0, cmp[1])

    # Equality
    qc.ccx(cmp[0], cmp[1], cmp[2])

In [9]:
def undo_compare_registers(qc, regA, regB, cmp):
    """
    Reverse compare_registers().
    """

    a1, a0 = regA
    b1, b0 = regB

    qc.ccx(cmp[0], cmp[1], cmp[2])

    qc.cx(b0, cmp[1])
    qc.cx(a0, cmp[1])

    qc.cx(b1, cmp[0])
    qc.cx(a1, cmp[0])

    qc.x(cmp[1])
    qc.x(cmp[0])

In [ ]:
def compare_constant(qc, cell_reg, value, anc):
    """
    anc = 1 iff cell_reg == value

    value must be:
        0 -> 00
        1 -> 01
        2 -> 10
    """

    msb, lsb = cell_reg

    bits = {
        0: (0, 0),
        1: (0, 1),
        2: (1, 0)
    }[value]

    if bits[0] == 0:
        qc.x(msb)

    if bits[1] == 0:
        qc.x(lsb)

    qc.ccx(msb, lsb, anc)

    if bits[1] == 0:
        qc.x(lsb)

    if bits[0] == 0:
        qc.x(msb)

In [11]:
def undo_compare_constant(qc, cell_reg, value, anc):
    """
    Reverse compare_constant().

    Restores the cell qubits and ancilla to their original state.
    """

    msb, lsb = cell_reg

    bits = {
        0: (0, 0),
        1: (0, 1),
        2: (1, 0)
    }[value]

    # Reverse order of operations

    if bits[0] == 0:
        qc.x(msb)

    if bits[1] == 0:
        qc.x(lsb)

    qc.ccx(msb, lsb, anc)

    if bits[1] == 0:
        qc.x(lsb)

    if bits[0] == 0:
        qc.x(msb)

In [12]:
# TODO: Use without eq ancillas
# qc = QuantumCircuit(
#     cells,
#     enc,
#     given,
#     cmp,
#     row,
#     col,
#     flags,
#     phase,
#     c
# )

In [13]:
def oracle(
    qc,
    cells,
    enc,
    given,
    cmp,
    eq,
    row,
    col,
    flags,
    phase,
):

    # ==================================================
    # Compute
    # ==================================================

    qc.barrier()

    # ---------------- Encoding Check ----------------

    qc.barrier(*cells, *enc)

    for i in range(9):
        encoding_check(
            qc,
            reg(cells, i),
            enc[i]
        )

    qc.barrier(*cells, *enc)

    # ---------------- Given Value Check ----------------

    qc.barrier(
        *reg(cells, 0),
        *reg(cells, 4),
        *reg(cells, 8),
        *given
    )

    compare_constant(
        qc,
        reg(cells, 0),
        0,
        given[0]
    )

    compare_constant(
        qc,
        reg(cells, 4),
        1,
        given[1]
    )

    compare_constant(
        qc,
        reg(cells, 8),
        2,
        given[2]
    )

    qc.barrier(
        *reg(cells, 0),
        *reg(cells, 4),
        *reg(cells, 8),
        *given
    )

    # ---------------- Row Constraints ----------------

    qc.barrier(*cells, *cmp, *eq, *row)

    # ==========================
    # Row 0 : (v0, v1, v2)
    # ==========================

    compare_registers(qc, reg(cells, 0), reg(cells, 1), cmp)
    qc.cx(cmp[2], eq[0])
    undo_compare_registers(qc, reg(cells, 0), reg(cells, 1), cmp)

    compare_registers(qc, reg(cells, 0), reg(cells, 2), cmp)
    qc.cx(cmp[2], eq[1])
    undo_compare_registers(qc, reg(cells, 0), reg(cells, 2), cmp)

    compare_registers(qc, reg(cells, 1), reg(cells, 2), cmp)
    qc.cx(cmp[2], eq[2])
    undo_compare_registers(qc, reg(cells, 1), reg(cells, 2), cmp)

    qc.x(eq[0])
    qc.x(eq[1])
    qc.x(eq[2])

    qc.mcx(eq[0:3], row[0])

    qc.x(eq[0])
    qc.x(eq[1])
    qc.x(eq[2])

    # ==========================
    # Row 1 : (v3, v4, v5)
    # ==========================

    compare_registers(qc, reg(cells, 3), reg(cells, 4), cmp)
    qc.cx(cmp[2], eq[3])
    undo_compare_registers(qc, reg(cells, 3), reg(cells, 4), cmp)

    compare_registers(qc, reg(cells, 3), reg(cells, 5), cmp)
    qc.cx(cmp[2], eq[4])
    undo_compare_registers(qc, reg(cells, 3), reg(cells, 5), cmp)

    compare_registers(qc, reg(cells, 4), reg(cells, 5), cmp)
    qc.cx(cmp[2], eq[5])
    undo_compare_registers(qc, reg(cells, 4), reg(cells, 5), cmp)

    qc.x(eq[3])
    qc.x(eq[4])
    qc.x(eq[5])

    qc.mcx(eq[3:6], row[1])

    qc.x(eq[3])
    qc.x(eq[4])
    qc.x(eq[5])

    # ==========================
    # Row 2 : (v6, v7, v8)
    # ==========================

    compare_registers(qc, reg(cells, 6), reg(cells, 7), cmp)
    qc.cx(cmp[2], eq[6])
    undo_compare_registers(qc, reg(cells, 6), reg(cells, 7), cmp)

    compare_registers(qc, reg(cells, 6), reg(cells, 8), cmp)
    qc.cx(cmp[2], eq[7])
    undo_compare_registers(qc, reg(cells, 6), reg(cells, 8), cmp)

    compare_registers(qc, reg(cells, 7), reg(cells, 8), cmp)
    qc.cx(cmp[2], eq[8])
    undo_compare_registers(qc, reg(cells, 7), reg(cells, 8), cmp)

    qc.x(eq[6])
    qc.x(eq[7])
    qc.x(eq[8])

    qc.mcx(eq[6:9], row[2])

    qc.x(eq[6])
    qc.x(eq[7])
    qc.x(eq[8])

    qc.barrier(*cells, *cmp, *eq, *row)

    # ---------------- Column Constraints ----------------

    qc.barrier(*cells, *cmp, *eq, *col)

    # ==========================
    # Column 0 : (v0, v3, v6)
    # ==========================

    compare_registers(qc, reg(cells,0), reg(cells,3), cmp)
    qc.cx(cmp[2], eq[9])
    undo_compare_registers(qc, reg(cells,0), reg(cells,3), cmp)

    compare_registers(qc, reg(cells,0), reg(cells,6), cmp)
    qc.cx(cmp[2], eq[10])
    undo_compare_registers(qc, reg(cells,0), reg(cells,6), cmp)

    compare_registers(qc, reg(cells,3), reg(cells,6), cmp)
    qc.cx(cmp[2], eq[11])
    undo_compare_registers(qc, reg(cells,3), reg(cells,6), cmp)

    qc.x(eq[9])
    qc.x(eq[10])
    qc.x(eq[11])

    qc.mcx(eq[9:12], col[0])

    qc.x(eq[9])
    qc.x(eq[10])
    qc.x(eq[11])

    # ==========================
    # Column 1 : (v1, v4, v7)
    # ==========================

    compare_registers(qc, reg(cells,1), reg(cells,4), cmp)
    qc.cx(cmp[2], eq[12])
    undo_compare_registers(qc, reg(cells,1), reg(cells,4), cmp)

    compare_registers(qc, reg(cells,1), reg(cells,7), cmp)
    qc.cx(cmp[2], eq[13])
    undo_compare_registers(qc, reg(cells,1), reg(cells,7), cmp)

    compare_registers(qc, reg(cells,4), reg(cells,7), cmp)
    qc.cx(cmp[2], eq[14])
    undo_compare_registers(qc, reg(cells,4), reg(cells,7), cmp)

    qc.x(eq[12])
    qc.x(eq[13])
    qc.x(eq[14])

    qc.mcx(eq[12:15], col[1])

    qc.x(eq[12])
    qc.x(eq[13])
    qc.x(eq[14])

    # ==========================
    # Column 2 : (v2, v5, v8)
    # ==========================

    compare_registers(qc, reg(cells,2), reg(cells,5), cmp)
    qc.cx(cmp[2], eq[15])
    undo_compare_registers(qc, reg(cells,2), reg(cells,5), cmp)

    compare_registers(qc, reg(cells,2), reg(cells,8), cmp)
    qc.cx(cmp[2], eq[16])
    undo_compare_registers(qc, reg(cells,2), reg(cells,8), cmp)

    compare_registers(qc, reg(cells,5), reg(cells,8), cmp)
    qc.cx(cmp[2], eq[17])
    undo_compare_registers(qc, reg(cells,5), reg(cells,8), cmp)

    qc.x(eq[15])
    qc.x(eq[16])
    qc.x(eq[17])

    qc.mcx(eq[15:18], col[2])

    qc.x(eq[15])
    qc.x(eq[16])
    qc.x(eq[17])

    qc.barrier(*cells, *cmp, *eq, *col)

    # ---------------- Global Constraint ----------------

    qc.barrier(*enc, *given, *row, *col, *flags)

    # All encodings valid
    qc.mcx(enc[:], flags[0])

    # All Sudoku constraints valid
    qc.mcx(
        [
            given[0], given[1], given[2],
            row[0], row[1], row[2],
            col[0], col[1], col[2]
        ],
        flags[1]
    )

    # Final oracle flag
    qc.ccx(flags[0], flags[1], flags[2])

    qc.barrier(*enc, *given, *row, *col, *flags)

    # ---------------- Phase Flip ----------------

    qc.barrier(*flags, *phase)

    qc.cx(flags[2], phase[0])

    qc.barrier(*flags, *phase)

    # ==================================================
    # Uncompute
    # ==================================================

    # ---------------- Undo Global Constraint ----------------

    qc.barrier(*enc, *given, *row, *col, *flags)

    qc.ccx(flags[0], flags[1], flags[2])

    qc.mcx(
        [
            given[0], given[1], given[2],
            row[0], row[1], row[2],
            col[0], col[1], col[2]
        ],
        flags[1]
    )

    qc.mcx(enc[:], flags[0])

    qc.barrier(*enc, *given, *row, *col, *flags)

    # ---------------- Undo Column Constraints ----------------

    qc.barrier(*cells, *cmp, *eq, *col)

    # ==========================
    # Undo Column 2
    # ==========================

    qc.x(eq[15])
    qc.x(eq[16])
    qc.x(eq[17])

    qc.mcx(eq[15:18], col[2])

    qc.x(eq[15])
    qc.x(eq[16])
    qc.x(eq[17])

    compare_registers(qc, reg(cells,5), reg(cells,8), cmp)
    qc.cx(cmp[2], eq[17])
    undo_compare_registers(qc, reg(cells,5), reg(cells,8), cmp)

    compare_registers(qc, reg(cells,2), reg(cells,8), cmp)
    qc.cx(cmp[2], eq[16])
    undo_compare_registers(qc, reg(cells,2), reg(cells,8), cmp)

    compare_registers(qc, reg(cells,2), reg(cells,5), cmp)
    qc.cx(cmp[2], eq[15])
    undo_compare_registers(qc, reg(cells,2), reg(cells,5), cmp)

    # ==========================
    # Undo Column 1
    # ==========================

    qc.x(eq[12])
    qc.x(eq[13])
    qc.x(eq[14])

    qc.mcx(eq[12:15], col[1])

    qc.x(eq[12])
    qc.x(eq[13])
    qc.x(eq[14])

    compare_registers(qc, reg(cells,4), reg(cells,7), cmp)
    qc.cx(cmp[2], eq[14])
    undo_compare_registers(qc, reg(cells,4), reg(cells,7), cmp)

    compare_registers(qc, reg(cells,1), reg(cells,7), cmp)
    qc.cx(cmp[2], eq[13])
    undo_compare_registers(qc, reg(cells,1), reg(cells,7), cmp)

    compare_registers(qc, reg(cells,1), reg(cells,4), cmp)
    qc.cx(cmp[2], eq[12])
    undo_compare_registers(qc, reg(cells,1), reg(cells,4), cmp)

    # ==========================
    # Undo Column 0
    # ==========================

    qc.x(eq[9])
    qc.x(eq[10])
    qc.x(eq[11])

    qc.mcx(eq[9:12], col[0])

    qc.x(eq[9])
    qc.x(eq[10])
    qc.x(eq[11])

    compare_registers(qc, reg(cells,3), reg(cells,6), cmp)
    qc.cx(cmp[2], eq[11])
    undo_compare_registers(qc, reg(cells,3), reg(cells,6), cmp)

    compare_registers(qc, reg(cells,0), reg(cells,6), cmp)
    qc.cx(cmp[2], eq[10])
    undo_compare_registers(qc, reg(cells,0), reg(cells,6), cmp)

    compare_registers(qc, reg(cells,0), reg(cells,3), cmp)
    qc.cx(cmp[2], eq[9])
    undo_compare_registers(qc, reg(cells,0), reg(cells,3), cmp)

    qc.barrier(*cells, *cmp, *eq, *col)

    # ---------------- Undo Row Constraints ----------------

    qc.barrier(*cells, *cmp, *eq, *row)

    # ==========================
    # Undo Row 2
    # ==========================

    qc.x(eq[6])
    qc.x(eq[7])
    qc.x(eq[8])

    qc.mcx(eq[6:9], row[2])

    qc.x(eq[6])
    qc.x(eq[7])
    qc.x(eq[8])

    compare_registers(qc, reg(cells, 7), reg(cells, 8), cmp)
    qc.cx(cmp[2], eq[8])
    undo_compare_registers(qc, reg(cells, 7), reg(cells, 8), cmp)

    compare_registers(qc, reg(cells, 6), reg(cells, 8), cmp)
    qc.cx(cmp[2], eq[7])
    undo_compare_registers(qc, reg(cells, 6), reg(cells, 8), cmp)

    compare_registers(qc, reg(cells, 6), reg(cells, 7), cmp)
    qc.cx(cmp[2], eq[6])
    undo_compare_registers(qc, reg(cells, 6), reg(cells, 7), cmp)

    # ==========================
    # Undo Row 1
    # ==========================

    qc.x(eq[3])
    qc.x(eq[4])
    qc.x(eq[5])

    qc.mcx(eq[3:6], row[1])

    qc.x(eq[3])
    qc.x(eq[4])
    qc.x(eq[5])

    compare_registers(qc, reg(cells, 4), reg(cells, 5), cmp)
    qc.cx(cmp[2], eq[5])
    undo_compare_registers(qc, reg(cells, 4), reg(cells, 5), cmp)

    compare_registers(qc, reg(cells, 3), reg(cells, 5), cmp)
    qc.cx(cmp[2], eq[4])
    undo_compare_registers(qc, reg(cells, 3), reg(cells, 5), cmp)

    compare_registers(qc, reg(cells, 3), reg(cells, 4), cmp)
    qc.cx(cmp[2], eq[3])
    undo_compare_registers(qc, reg(cells, 3), reg(cells, 4), cmp)

    # ==========================
    # Undo Row 0
    # ==========================

    qc.x(eq[0])
    qc.x(eq[1])
    qc.x(eq[2])

    qc.mcx(eq[0:3], row[0])

    qc.x(eq[0])
    qc.x(eq[1])
    qc.x(eq[2])

    compare_registers(qc, reg(cells, 1), reg(cells, 2), cmp)
    qc.cx(cmp[2], eq[2])
    undo_compare_registers(qc, reg(cells, 1), reg(cells, 2), cmp)

    compare_registers(qc, reg(cells, 0), reg(cells, 2), cmp)
    qc.cx(cmp[2], eq[1])
    undo_compare_registers(qc, reg(cells, 0), reg(cells, 2), cmp)

    compare_registers(qc, reg(cells, 0), reg(cells, 1), cmp)
    qc.cx(cmp[2], eq[0])
    undo_compare_registers(qc, reg(cells, 0), reg(cells, 1), cmp)

    qc.barrier(*cells, *cmp, *eq, *row)

    # ---------------- Undo Given Value Check ----------------

    qc.barrier(
        *reg(cells, 0),
        *reg(cells, 4),
        *reg(cells, 8),
        *given
    )

    undo_compare_constant(
        qc,
        reg(cells, 8),
        2,
        given[2]
    )

    undo_compare_constant(
        qc,
        reg(cells, 4),
        1,
        given[1]
    )

    undo_compare_constant(
        qc,
        reg(cells, 0),
        0,
        given[0]
    )

    qc.barrier(
        *reg(cells, 0),
        *reg(cells, 4),
        *reg(cells, 8),
        *given
    )

    # ---------------- Undo Encoding Check ----------------

    qc.barrier(*cells, *enc)

    for i in reversed(range(9)):
        undo_encoding_check(
            qc,
            reg(cells, i),
            enc[i]
        )

    qc.barrier(*cells, *enc)

In [14]:
cells = QuantumRegister(18, "cell")
enc = QuantumRegister(9,  "enc")
given = QuantumRegister(3, "given")
cmp = QuantumRegister(3,   "cmp")
# TODO: Merge 9 row and column check eq later to cmp directly
eq = QuantumRegister(18,    "eq")
row = QuantumRegister(3,   "row")
col = QuantumRegister(3,   "col")
flags = QuantumRegister(3, "flags")
phase = QuantumRegister(1, "phase")

c = ClassicalRegister(18, "c")
f2 = ClassicalRegister(1, "f2")

In [15]:
qc = QuantumCircuit(
    cells,
    enc,
    given,
    cmp,
    eq,
    row,
    col,
    flags,
    phase,
    c,
    f2
)

In [16]:
initialize_search_space(qc, cells)

qc.barrier()

initialize_phase_ancilla(qc, phase)

oracle(
        qc,
        cells,
        enc,
        given,
        cmp,
        eq,
        row,
        col,
        flags,
        phase,
)

qc.barrier()

# qc.measure(cells, c)
# qc.measure(flags[2], f2)

CircuitInstruction(operation=Instruction(name='barrier', num_qubits=61, num_clbits=0, params=[]), qubits=(<Qubit register=(18, "cell"), index=0>, <Qubit register=(18, "cell"), index=1>, <Qubit register=(18, "cell"), index=2>, <Qubit register=(18, "cell"), index=3>, <Qubit register=(18, "cell"), index=4>, <Qubit register=(18, "cell"), index=5>, <Qubit register=(18, "cell"), index=6>, <Qubit register=(18, "cell"), index=7>, <Qubit register=(18, "cell"), index=8>, <Qubit register=(18, "cell"), index=9>, <Qubit register=(18, "cell"), index=10>, <Qubit register=(18, "cell"), index=11>, <Qubit register=(18, "cell"), index=12>, <Qubit register=(18, "cell"), index=13>, <Qubit register=(18, "cell"), index=14>, <Qubit register=(18, "cell"), index=15>, <Qubit register=(18, "cell"), index=16>, <Qubit register=(18, "cell"), index=17>, <Qubit register=(9, "enc"), index=0>, <Qubit register=(9, "enc"), index=1>, <Qubit register=(9, "enc"), index=2>, <Qubit register=(9, "enc"), index=3>, <Qubit registe

In [17]:
qc.draw(
    "mpl",
    idle_wires=False,
    fold=-1
)

In [18]:
# from qiskit_aer import AerSimulator
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
# from qiskit_ibm_runtime import (
#     SamplerV2 as Sampler,
# )

In [19]:
# print(qc.count_ops())

In [20]:
# from qiskit import transpile

In [21]:
# sim = AerSimulator(
#     method="tensor_network",
#     device="GPU"
# )

# # pm = generate_preset_pass_manager(backend=sim, optimization_level=3)
# isa_circuit = transpile(qc,
#                         optimization_level=3
#                         )

# sampler = Sampler(mode=sim)

# result = sampler.run([isa_circuit], shots=2).result()

# counts = result.get_counts()

# print(counts)